# 02 — Preprocessing

Clean sensors, split **by unit** (no leakage), scale, and persist processed tables.


In [ ]:
import sys
from pathlib import Path
ROOT = Path.cwd() if (Path.cwd() / "src").exists() else Path.cwd().parent
sys.path.insert(0, str(ROOT))
print("Project root:", ROOT)

In [ ]:
from src.data_preprocessing import (
    SENSOR_COLUMNS,
    clean_sensor_frame,
    load_raw,
    add_rul_if_missing,
    clip_rul,
    split_by_unit,
    fit_scaler,
    apply_scaler,
    save_processed,
    prepare_datasets,
)

raw = add_rul_if_missing(clean_sensor_frame(load_raw()))
raw = clip_rul(raw, max_rul=125)
raw.head()

In [ ]:
train, test = split_by_unit(raw)
print("Train units", train["unit_id"].nunique(), "rows", len(train))
print("Test units", test["unit_id"].nunique(), "rows", len(test))
assert set(train["unit_id"]).isdisjoint(set(test["unit_id"]))

In [ ]:
scaler = fit_scaler(train)
train_s = apply_scaler(train, scaler)
test_s = apply_scaler(test, scaler)
train_s[SENSOR_COLUMNS].describe().loc[["min", "max"]]

In [ ]:
save_processed(train_s, "train.csv")
save_processed(test_s, "test.csv")

# Equivalent one-shot helper used by the dashboard:
train_s, test_s, scaler = prepare_datasets()
train_s.shape, test_s.shape